# PF00042: complete protein-set-to-graph pipeline

This notebook completes the required pipeline before data exploration:

```text
protein FASTA -> Biopython / BLAST / DEDAL pair scores -> quality record -> separate graphs
```

The alignment method, graph threshold rule, graph threshold, and correlation-check thresholds are explicit configuration values. The example graph thresholds below are only software smoke-test settings. They are not scientific choices and will be decided after exploration in notebook 03.

In [ ]:
import subprocess
import sys
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIRECTORY = PROJECT_ROOT / "data/processed/pfam_pf00042_globin_pilot"
FASTA_PATH = DATASET_DIRECTORY / "PF00042_globin_pilot.fasta"
METADATA_PATH = DATASET_DIRECTORY / "PF00042_globin_pilot_metadata.tsv"
SCORE_DIRECTORY = PROJECT_ROOT / "outputs/tables/pfam_pf00042_correlation"
GRAPH_DIRECTORY = PROJECT_ROOT / "outputs/graphs/pfam_pf00042_pipeline"

## 1. Pipeline configuration

Each algorithm produces its own graph. `top_n=13` retains the strongest 13 available pairs for each method, giving equal edge counts for a simple pipeline check. Change the rule to `absolute` or `percentile` as required later.

Correlation thresholds intentionally remain `None`: the pipeline records correlations and overlap but does not label them good or poor until notebook 03 provides evidence for suitable cutoffs.

In [ ]:
RUN_SCORING = True
METHODS_TO_RUN = ("biopython", "blast", "dedal")

GRAPH_SPECS = {
    "biopython": {"rule": "top_n", "threshold": 13},
    "blast": {"rule": "top_n", "threshold": 13},
    "dedal": {"rule": "top_n", "threshold": 13},
}

CORRELATION_CHECKS = {
    "minimum_rho": None,
    "minimum_overlap_fraction": None,
    "minimum_pairs": None,
}

## 2. Load the protein set

The FASTA supplies the protein identifiers and sequences. The metadata table supplies node attributes such as subgroup, organism, taxonomy, and domain length.

In [ ]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
protein_ids = metadata["uniprot_accession"].tolist()
expected_pair_count = len(protein_ids) * (len(protein_ids) - 1) // 2
print(f"Proteins: {len(protein_ids)}")
print(f"Unique pairs: {expected_pair_count}")
display(metadata[["uniprot_accession", "organism", "subgroup", "domain_length"]])

## 3. Produce or reuse method scores

The scoring runner saves one canonical row for every unique protein pair. Existing outputs are reused, so DEDAL is not recalculated unnecessarily. On a new protein set, including `dedal` will run its slower local model.

In [ ]:
if RUN_SCORING:
    scoring_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/run_pf00042_correlation.py"),
        "--fasta",
        str(FASTA_PATH),
        "--output-directory",
        str(SCORE_DIRECTORY),
        "--methods",
        *METHODS_TO_RUN,
        "--threads",
        "4",
    ]
    subprocess.run(scoring_command, cwd=PROJECT_ROOT, check=True)

paired_scores = pd.read_csv(SCORE_DIRECTORY / "paired_scores.tsv", sep="\t")
assert len(paired_scores) == expected_pair_count
display(paired_scores.head())

## 4. Build separate algorithm graphs

The graph runner applies each method's own configured rule and writes separate GraphML, edge-table, node-table, summary, and provenance files. Missing BLAST hits remain unavailable and can never become zero-score edges.

In [ ]:
graph_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/build_method_graphs.py"),
    "--fasta",
    str(FASTA_PATH),
    "--pairs",
    str(SCORE_DIRECTORY / "paired_scores.tsv"),
    "--metadata",
    str(METADATA_PATH),
    "--correlations",
    str(SCORE_DIRECTORY / "spearman_correlations.tsv"),
    "--output-directory",
    str(GRAPH_DIRECTORY),
]
for method, specification in GRAPH_SPECS.items():
    graph_command.extend(
        [
            "--graph",
            f"{method}:{specification['rule']}:{specification['threshold']}",
        ]
    )
for option, value in CORRELATION_CHECKS.items():
    if value is not None:
        graph_command.extend([f"--{option.replace('_', '-')}", str(value)])
subprocess.run(graph_command, cwd=PROJECT_ROOT, check=True)

## 5. Verify graph outputs

The smoke-test configuration should create three different graph objects with the complete protein set. The quality table records evidence but remains `not_evaluated` until explicit correlation thresholds are configured.

In [ ]:
summaries = pd.read_csv(GRAPH_DIRECTORY / "graph_summaries.tsv", sep="\t")
quality = pd.read_csv(GRAPH_DIRECTORY / "quality_flags.tsv", sep="\t")
graphs = {}
for method, specification in GRAPH_SPECS.items():
    label = str(specification["threshold"]).replace(".", "p")
    path = GRAPH_DIRECTORY / f"{method}_{specification['rule']}_{label}.graphml"
    graphs[method] = nx.read_graphml(path)
    assert set(graphs[method]) == set(protein_ids)

display(summaries)
display(quality)

## Pipeline status

The pipeline now accepts a protein set, runs selectable alignment methods, records configurable correlation checks, and emits a separate saved graph for each method under explicit threshold settings. Notebook 03 is the next stage: explore the data and results, then replace these smoke-test graph settings with justified thresholds and sensitivity ranges.